# AIN433 Assignment 2: Homography Estimation and Panorama Stitching
Student Number: 2220765033

- Feature detection and matching (SIFT/ORB)
- Manual DLT + RANSAC homography estimation
- Panorama stitching for all 6 scenes
- Augmented Reality application
- Comprehensive visualizations and experiments

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab import drive
import os
import time
from scipy.ndimage import distance_transform_edt
import glob

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Set paths
BASE_PATH = '/content/drive/MyDrive/CV_Assignment2/pa2_data/'
PANORAMA_PATH = BASE_PATH + 'panorama_dataset/'
AR_PATH = BASE_PATH + 'ar_dataset/'
OUTPUT_PATH = BASE_PATH + 'results/'

# Create output directories
os.makedirs(OUTPUT_PATH + 'panoramas/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'visualizations/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'keypoints/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'matches/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'inliers/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'warped/', exist_ok=True)
os.makedirs(OUTPUT_PATH + 'ar_frames/', exist_ok=True)

In [ ]:
class FeatureDetector:
    """
    Feature detection using SIFT or ORB

    SIFT: Scale-Invariant Feature Transform
    - Detects blobs at multiple scales using Difference-of-Gaussians (DoG)
    - Produces 128-dimensional floating-point descriptors
    - Invariant to scale, rotation, and partially to illumination changes
    - Best for scenes with significant scale/rotation variations

    ORB: Oriented FAST and Rotated BRIEF
    - Uses FAST corner detector for keypoint localization
    - Computes 256-bit binary descriptors using rBRIEF
    - Much faster than SIFT but less robust to scale changes
    - Suitable for real-time applications
    """

    def __init__(self, method='SIFT'):
        """
        Initialize feature detector
        Args: method: 'SIFT' or 'ORB'
        """
        self.method = method

        if method == 'SIFT':
            # SIFT with default parameters:
            self.detector = cv2.SIFT_create()

        elif method == 'ORB':
            # ORB with 2000 features for sufficient coverage
            self.detector = cv2.ORB_create(nfeatures=2000)

        else:
            raise ValueError("Method must be 'SIFT' or 'ORB'")

    def detect_and_compute(self, image):
        """
        Detect keypoints and compute descriptors

        Args:image: Grayscale input image

        Returns:
            keypoints: List of cv2.KeyPoint objects
            descriptors: NumPy array of feature descriptors
        """
        keypoints, descriptors = self.detector.detectAndCompute(image, None)
        return keypoints, descriptors

    def visualize_keypoints(self, image, keypoints, title="Keypoints", save_path=None):
        """
        Visualize detected keypoints with rich information (size, orientation)

        Args:
            image: Input image (BGR)
            keypoints: Detected keypoints
            title: Plot title
            save_path: Optional path to save visualization
        """
        # Draw keypoints with size and orientation indicators
        img_kp = cv2.drawKeypoints(
            image, keypoints, None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )

        plt.figure(figsize=(14, 9))
        plt.imshow(cv2.cvtColor(img_kp, cv2.COLOR_BGR2RGB))
        plt.title(f"{title} - {len(keypoints)} keypoints ({self.method})")
        plt.axis('off')

        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

        # Report keypoint statistics
        print(f" Keypoints detected: {len(keypoints)}")
        print(f" Spatial coverage: {'Good' if len(keypoints) > 500 else 'Limited'}")


class FeatureMatcher:
    """
    Feature matching with Lowe's ratio test

    Lowe's Ratio Test:
    - For each descriptor in image 1, find k=2 nearest neighbors in image 2
    - Accept match if distance(nearest) < ratio_thresh * distance(second_nearest)
    - Filters ambiguous matches where multiple similar descriptors exist
    - Typical ratio_thresh: 0.7-0.8 (lower = stricter filtering)
    """

    def __init__(self, method='SIFT', ratio_thresh=0.75):
        """
        Initialize feature matcher
        Args:
            method: 'SIFT' or 'ORB' (determines distance metric)
            ratio_thresh: Lowe's ratio test threshold
        """
        self.method = method
        self.ratio_thresh = ratio_thresh

        if method == 'SIFT':
            # L2 (Euclidean) distance for SIFT's continuous descriptors
            self.matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        else:
            # Hamming distance for ORB's binary descriptors
            self.matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

    def match_features(self, desc1, desc2):
        """
        Match features using k-NN and Lowe's ratio test
        aargs:
            desc1: Descriptors from first image
            desc2: Descriptors from second image

        Returns:
            good_matches: List of filtered cv2.DMatch objects
        """
        # Find k=2 nearest neighbors for each descriptor
        matches = self.matcher.knnMatch(desc1, desc2, k=2)

        # Apply Lowe's ratio test
        good_matches = []
        for match_pair in matches:
            # Some descriptors may have fewer than 2 neighbors
            if len(match_pair) == 2:
                m, n = match_pair  # nearest and second-nearest
                if m.distance < self.ratio_thresh * n.distance:
                    good_matches.append(m)

        return good_matches

    def visualize_matches(self, img1, kp1, img2, kp2, matches, title="Matches", save_path=None):
        """
        Visualize feature correspondences between image pairs

        Args:
            img1, img2: Input images
            kp1, kp2: Keypoints
            matches: List of matches
            title: Plot title
            save_path: Optional save path
        """
        img_matches = cv2.drawMatches(
            img1, kp1, img2, kp2, matches, None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        plt.figure(figsize=(18, 9))
        plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
        plt.title(f"{title} - {len(matches)} matches")
        plt.axis('off')

        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

        print(f" Matches after ratio test: {len(matches)}")


In [ ]:
# PART 2: HOMOGRAPHY ESTIMATION - DLT + RANSAC
class HomographyEstimator:
    """
    Manual implementation of Direct Linear Transform (DLT) and RANSAC

    DLT Algorithm:
    - Solves for 3x3 homography matrix H from point correspondences
    - Each correspondence (x,y) -> (x',y') provides 2 linear equations
    - Minimum 4 correspondences needed (8 equations for 8 unknowns, h33=1)
    - Uses SVD to find least-squares solution: H = argmin ||Ah||

    RANSAC (RANdom SAmple Consensus):
    - Robust estimation in presence of outliers
    - Iteratively samples minimal sets (4 points) to estimate H
    - Counts inliers (reprojection error < threshold)
    - Returns model with maximum inlier support
    - Refines final H using all inliers
    """

    def __init__(self, ransac_thresh=5.0, ransac_iterations=2000, seed=42):
        """
        Initialize homography estimator

        Args:
            ransac_thresh: Inlier threshold in pixels (reprojection error)
            ransac_iterations: Number of RANSAC iterations
            seed: Random seed for reproducibility
        """
        self.ransac_thresh = float(ransac_thresh)
        self.ransac_iterations = int(ransac_iterations)
        self.rng = np.random.default_rng(seed)

    def normalize_points(self, points):
        """
        Normalize points for numerical stability (Hartley normalization)

        Why normalize?
        - Improves conditioning of matrix A in DLT
        - Prevents numerical errors from dominating the solution
        - Standard practice: translate centroid to origin, scale mean distance to √2

        Args:
            points: Nx2 array of 2D points

        Returns:
            pts_n: Normalized points
            T: 3x3 normalization transformation matrix
        """
        points = np.asarray(points, dtype=np.float64)

        # Compute centroid
        centroid = points.mean(axis=0)

        # Shift points to center
        shifted = points - centroid

        # Compute mean distance from origin
        dists = np.sqrt(shifted[:, 0]**2 + shifted[:, 1]**2)
        mean_dist = np.mean(dists) + 1e-12  # avoid division by zero

        # Scale factor to make mean distance = √2
        scale = np.sqrt(2.0) / mean_dist

        # Construct normalization matrix T
        T = np.array([
            [scale, 0, -scale * centroid[0]],
            [0, scale, -scale * centroid[1]],
            [0, 0, 1]
        ], dtype=np.float64)

        # Apply normalization: pts_normalized = T @ pts_homogeneous
        ones = np.ones((points.shape[0], 1))
        pts_h = np.hstack([points, ones])
        pts_n = (T @ pts_h.T).T
        pts_n = pts_n[:, :2] / pts_n[:, [2]]  # convert back to 2D

        return pts_n, T

    def compute_homography_dlt(self, src_pts, dst_pts):
        """
        Compute homography using Direct Linear Transform (DLT)
        Args:
            src_pts: Nx2 source points
            dst_pts: Nx2 destination points

        Returns:
            H: 3x3 homography matrix
        """
        src_pts = np.asarray(src_pts, dtype=np.float64)
        dst_pts = np.asarray(dst_pts, dtype=np.float64)

        # Normalize points for numerical stability
        src_n, T_src = self.normalize_points(src_pts)
        dst_n, T_dst = self.normalize_points(dst_pts)

        N = src_n.shape[0]

        # Construct matrix A (2N x 9)
        A = np.zeros((2*N, 9), dtype=np.float64)
        for i in range(N):
            x, y = src_n[i]
            xp, yp = dst_n[i]

            # Two rows per correspondence
            A[2*i] = [-x, -y, -1, 0, 0, 0, x*xp, y*xp, xp]
            A[2*i+1] = [0, 0, 0, -x, -y, -1, x*yp, y*yp, yp]

        # Solve Ah = 0 using SVD: A = U Σ V^T
        # Solution h is last column of V (corresponds to smallest singular value)
        _, _, Vt = np.linalg.svd(A)
        h = Vt[-1]
        H_norm = h.reshape(3, 3)

        # Denormalize: H = T_dst^(-1) @ H_norm @ T_src
        H = np.linalg.inv(T_dst) @ H_norm @ T_src

        # Normalize so that h33 = 1
        if abs(H[2, 2]) > 1e-12:
            H = H / H[2, 2]

        return H

    def compute_reprojection_error(self, src_pts, dst_pts, H):
        """
        Compute reprojection error for each correspondence

        Reprojection error = || dst_pts - H @ src_pts ||
        Measures how well H maps source points to destination points

        Args:
            src_pts: Nx2 source points
            dst_pts: Nx2 destination points
            H: 3x3 homography matrix

        Returns:
            errors: N-dimensional array of Euclidean distances (pixels)
        """
        src_pts = np.asarray(src_pts, dtype=np.float64)
        dst_pts = np.asarray(dst_pts, dtype=np.float64)

        # Convert to homogeneous coordinates
        ones = np.ones((src_pts.shape[0], 1))
        src_h = np.hstack([src_pts, ones])

        # Project: proj = H @ src
        proj_h = (H @ src_h.T).T
        proj = proj_h[:, :2] / (proj_h[:, [2]] + 1e-12)

        # Compute Euclidean distance
        errors = np.sqrt(np.sum((proj - dst_pts)**2, axis=1))

        return errors

    def ransac_homography(self, src_pts, dst_pts):
        """
        Robust homography estimation using RANSAC

        RANSAC Algorithm:
        1. Randomly sample 4 correspondences (minimal set)
        2. Compute homography H using DLT
        3. Count inliers: points with reprojection error < threshold
        4. Repeat for N iterations, keep H with most inliers
        5. Refine H using all inliers (optional but recommended)

        Number of iterations N chosen to ensure with probability p (e.g., 0.99)
        that at least one sample is outlier-free:
            N = log(1-p) / log(1-(1-ε)^s)
        where ε = outlier ratio, s = sample size (4 for homography)

        For ε=0.5, p=0.99, s=4: N ≈ 72 iterations
        Using N=2000 provides high confidence even with 70% outliers

        Args:
            src_pts: Nx2 source points
            dst_pts: Nx2 destination points

        Returns:
            best_H: Best homography matrix
            best_inliers: Boolean mask of inlier correspondences
        """
        src_pts = np.asarray(src_pts, dtype=np.float64)
        dst_pts = np.asarray(dst_pts, dtype=np.float64)
        N = src_pts.shape[0]

        best_H = None
        best_inliers = None
        best_count = 0

        # RANSAC iterations
        for iteration in range(self.ransac_iterations):
            # Step 1: Randomly sample 4 correspondences
            idx = self.rng.choice(N, size=4, replace=False)

            # Step 2: Compute homography from minimal set
            try:
                H_cand = self.compute_homography_dlt(src_pts[idx], dst_pts[idx])
            except:
                # DLT can fail for degenerate configurations (collinear points)
                continue

            # Step 3: Compute reprojection errors for all points
            errors = self.compute_reprojection_error(src_pts, dst_pts, H_cand)

            # Step 4: Count inliers (error < threshold)
            inliers = errors < self.ransac_thresh
            count = np.sum(inliers)

            # Step 5: Update best model if more inliers found
            if count > best_count:
                best_count = count
                best_inliers = inliers
                best_H = H_cand

        if best_H is None:
            raise RuntimeError("RANSAC failed to find valid homography")

        # Step 6: Refine homography using all inliers (least-squares)
        try:
            best_H = self.compute_homography_dlt(
                src_pts[best_inliers],
                dst_pts[best_inliers]
            )
        except:
            # If refinement fails, keep initial estimate
            pass

        return best_H, best_inliers

    def visualize_inliers(self, img1, kp1, img2, kp2, matches, inlier_mask, save_path=None):
        """
        Visualize RANSAC inliers vs outliers

        - Green lines: inliers (geometrically consistent)
        - Red lines: outliers (rejected by RANSAC)

        Args:
            img1, img2: Input images
            kp1, kp2: Keypoints
            matches: All matches
            inlier_mask: Boolean array indicating inliers
            save_path: Optional save path
        """
        # Separate inliers and outliers
        inlier_matches = [m for i, m in enumerate(matches) if inlier_mask[i]]
        outlier_matches = [m for i, m in enumerate(matches) if not inlier_mask[i]]

        # Draw inliers in green
        img_result = cv2.drawMatches(
            img1, kp1, img2, kp2, inlier_matches, None,
            matchColor=(0, 255, 0),  # green
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        plt.figure(figsize=(18, 9))
        plt.imshow(cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB))

        inlier_count = len(inlier_matches)
        total = len(matches)
        inlier_pct = 100 * inlier_count / max(1, total)

        plt.title(f"RANSAC Results: {inlier_count}/{total} inliers ({inlier_pct:.1f}%)")
        plt.axis('off')

        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

        print(f"✓ Inliers: {inlier_count}/{total} ({inlier_pct:.1f}%)")

In [ ]:
# PART 3: IMAGE WARPING AND PANORAMA STITCHING
class PanoramaStitcher:
    """
    Image warping and panorama construction with multiple blending strategies

    Pipeline:
    1. Compute canvas size by projecting image corners
    2. Warp target image using homography
    3. Place reference image on canvas
    4. Blend overlapping regions

    Blending Methods:
    - 'copy': Simple replacement (visible seams)
    - 'average': 50/50 blend in overlap (ghosting artifacts)
    - 'linear': Distance-transform weighted blend (seamless)
    """

    def compute_canvas_size(self, img1_shape, img2_shape, H):
        """
        Compute panorama canvas dimensions and translation offset

        Strategy:
        - Project all 4 corners of both images
        - Find bounding box [x_min, y_min, x_max, y_max]
        - Canvas size = (x_max - x_min, y_max - y_min)
        - Offset translation ensures all coordinates are positive

        Args:
            img1_shape: Shape of reference image (h, w, c)
            img2_shape: Shape of target image
            H: Homography mapping img2 -> img1 frame

        Returns:
            canvas_size: (height, width) tuple
            offset: 3x3 translation matrix
        """
        h1, w1 = img1_shape[:2]
        h2, w2 = img2_shape[:2]

        # Define image corners in homogeneous coordinates
        corners1 = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]])
        corners2 = np.float32([[0, 0], [w2, 0], [w2, h2], [0, h2]])

        # Warp img2 corners using homography
        ones = np.ones((4, 1))
        corners2_h = np.hstack([corners2, ones])
        warped2 = (H @ corners2_h.T).T
        warped2 = warped2[:, :2] / warped2[:, [2]]  # normalize

        # Compute bounding box of all corners
        all_corners = np.vstack([corners1, warped2])
        [x_min, y_min] = np.floor(all_corners.min(axis=0)).astype(int)
        [x_max, y_max] = np.ceil(all_corners.max(axis=0)).astype(int)

        width = x_max - x_min
        height = y_max - y_min

        # Translation matrix to shift all coordinates to positive range
        offset = np.array([
            [1, 0, -x_min],
            [0, 1, -y_min],
            [0, 0, 1]
        ], dtype=np.float64)

        return (height, width), offset

    def blend_images(self, img1, img2, method='linear'):
        """
        Blend overlapping regions using specified method

        Args:
            img1: First image on canvas
            img2: Second image (warped) on canvas
            method: Blending strategy

        Returns:
            blended: Blended panorama image
        """
        # Create binary masks indicating valid pixels
        mask1 = (cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY) > 0).astype(np.float32)
        mask2 = (cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY) > 0).astype(np.float32)

        if method == 'linear':
            # Distance transform: distance to nearest zero pixel
            d1 = distance_transform_edt(mask1 > 0)
            d2 = distance_transform_edt(mask2 > 0)

            # Compute normalized weights
            w1 = d1 / (d1 + d2 + 1e-6)  # avoid division by zero
            w2 = d2 / (d1 + d2 + 1e-6)

            # Weighted blend
            blended = (img1.astype(float) * w1[..., None] +
                      img2.astype(float) * w2[..., None])
            return np.clip(blended, 0, 255).astype(np.uint8)

        elif method == 'average':
            # Identify overlapping regions
            overlap = (mask1 > 0) & (mask2 > 0)

            result = img1.copy().astype(float)
            result[~overlap] += img2[~overlap]  # non-overlapping: simple addition
            result[overlap] = 0.5 * img1[overlap] + 0.5 * img2[overlap]  # 50/50 blend

            return np.clip(result, 0, 255).astype(np.uint8)

        else:  # 'copy'
            # Simple replacement: img2 overwrites img1
            result = img1.copy()
            result[mask2 > 0] = img2[mask2 > 0]
            return result

    def create_panorama(self, img1, img2, H, blend_method='linear'):
        """
        Create panorama from two images using homography

        Complete stitching pipeline:
        1. Compute canvas size and offset
        2. Warp img2 to img1's coordinate frame
        3. Place img1 on canvas
        4. Blend overlapping regions

        Args:
            img1: Reference image (fixed)
            img2: Target image (to be warped)
            H: Homography matrix (img2 -> img1)
            blend_method: Blending strategy

        Returns:
            panorama: Final stitched image
        """
        # Step 1: Compute canvas dimensions and offset
        canvas_size, offset = self.compute_canvas_size(img1.shape, img2.shape, H)

        # Combine offset with homography
        H_offset = offset @ H

        # Step 2: Warp img2 using combined transformation
        warped_img2 = cv2.warpPerspective(
            img2, H_offset,
            (canvas_size[1], canvas_size[0])  # (width, height)
        )

        # Step 3: Place img1 on canvas at correct position
        canvas = np.zeros((canvas_size[0], canvas_size[1], 3), dtype=np.uint8)
        y_off = int(offset[1, 2])
        x_off = int(offset[0, 2])
        canvas[y_off:y_off+img1.shape[0], x_off:x_off+img1.shape[1]] = img1

        # Step 4: Blend images
        panorama = self.blend_images(canvas, warped_img2, method=blend_method)

        return panorama

In [ ]:
# PROCESS ALL 6 SCENES
def process_scene_complete(scene_name, method='SIFT', blend='linear', save_visualizations=True):
    """
    Process a single panorama scene with full visualization pipeline

    Args:
        scene_name: Scene folder name (e.g., 'v_bird')
        method: Feature detector ('SIFT' or 'ORB')
        blend: Blending method ('linear', 'average', 'copy')
        save_visualizations: Whether to save intermediate results

    Returns:
        Dictionary with processing statistics
    """
    print(f"Processing Scene: {scene_name}")

    # Load images from scene folder
    scene_path = os.path.join(PANORAMA_PATH, scene_name)
    image_files = []
    for ext in ['.ppm', '.png', '.jpg', '.jpeg']:
        image_files.extend(sorted(glob.glob(os.path.join(scene_path, f'*{ext}'))))

    if len(image_files) < 2:
        raise FileNotFoundError(f"Need at least 2 images in {scene_path}")

    print(f"Found {len(image_files)} images")

    # Load first two images
    img1 = cv2.imread(image_files[0])
    img2 = cv2.imread(image_files[1])
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Feature Detection
    print("\n[1/4] Feature Detection...")
    detector = FeatureDetector(method=method)
    kp1, desc1 = detector.detect_and_compute(gray1)
    kp2, desc2 = detector.detect_and_compute(gray2)

    if save_visualizations:
        save_path = os.path.join(OUTPUT_PATH, 'keypoints', f'{scene_name}_kp1.jpg')
        detector.visualize_keypoints(img1, kp1, f"{scene_name} - Image 1", save_path)

    #Feature Matching
    print("\n[2/4] Feature Matching...")
    matcher = FeatureMatcher(method=method, ratio_thresh=0.75)
    matches = matcher.match_features(desc1, desc2)

    if save_visualizations:
        save_path = os.path.join(OUTPUT_PATH, 'matches', f'{scene_name}_matches.jpg')
        matcher.visualize_matches(img1, kp1, img2, kp2, matches, f"{scene_name} - Matches", save_path)

    # Extract matched point coordinates
    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

    # Homography + RANSAC
    print("\n[3/4] Homography Estimation (DLT + RANSAC)...")
    estimator = HomographyEstimator(ransac_thresh=5.0, ransac_iterations=2000)
    H, inliers = estimator.ransac_homography(pts1, pts2)

    # Compute statistics
    inlier_count = np.sum(inliers)
    inlier_ratio = inlier_count / len(matches)
    errors = estimator.compute_reprojection_error(pts1[inliers], pts2[inliers], H)
    mean_error = np.mean(errors)

    print(f"✓ Inliers: {inlier_count}/{len(matches)} ({100*inlier_ratio:.1f}%)")
    print(f"✓ Mean reprojection error: {mean_error:.2f} pixels")

    if save_visualizations:
        save_path = os.path.join(OUTPUT_PATH, 'inliers', f'{scene_name}_inliers.jpg')
        estimator.visualize_inliers(img1, kp1, img2, kp2, matches, inliers, save_path)

    # Panorama Creation
    print("\n[4/4] Creating Panorama...")
    stitcher = PanoramaStitcher()

    # Note: We use H_inv because we want to warp img2 INTO img1's frame
    H_inv = np.linalg.inv(H)
    panorama = stitcher.create_panorama(img1, img2, H_inv, blend_method=blend)

    if save_visualizations:
        # Visualize intermediate warping results
        canvas_size, offset = stitcher.compute_canvas_size(img1.shape, img2.shape, H_inv)
        H_offset = offset @ H_inv
        warped_img2 = cv2.warpPerspective(img2, H_offset, (canvas_size[1], canvas_size[0]))

        canvas = np.zeros((canvas_size[0], canvas_size[1], 3), dtype=np.uint8)
        y_off = int(offset[1, 2])
        x_off = int(offset[0, 2])
        canvas[y_off:y_off+img1.shape[0], x_off:x_off+img1.shape[1]] = img1

        # Show canvas and warped image side by side
        plt.figure(figsize=(16, 8))
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        plt.title("Reference Image on Canvas")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(warped_img2, cv2.COLOR_BGR2RGB))
        plt.title("Warped Target Image")
        plt.axis('off')

        save_path = os.path.join(OUTPUT_PATH, 'warped', f'{scene_name}_warped.jpg')
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

        # Visualize overlap mask
        mask_canvas = (cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY) > 0)
        mask_warped = (cv2.cvtColor(warped_img2, cv2.COLOR_BGR2GRAY) > 0)
        overlap = (mask_canvas & mask_warped).astype(np.uint8) * 255

        plt.figure(figsize=(12, 6))
        plt.imshow(overlap, cmap='gray')
        plt.title("Overlapping Region (White)")
        plt.axis('off')
        save_path = os.path.join(OUTPUT_PATH, 'warped', f'{scene_name}_overlap.jpg')
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

    # Display final panorama
    plt.figure(figsize=(18, 9))
    plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
    plt.title(f"{scene_name} - Final Panorama ({blend} blending)")
    plt.axis('off')

    save_path = os.path.join(OUTPUT_PATH, 'panoramas', f'{scene_name}_panorama.jpg')
    cv2.imwrite(save_path, panorama)
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()

    print(f"✓ Panorama saved to: {save_path}")

    return {
        'scene': scene_name,
        'method': method,
        'keypoints': (len(kp1), len(kp2)),
        'matches': len(matches),
        'inliers': inlier_count,
        'inlier_ratio': inlier_ratio,
        'mean_error': mean_error,
        'blend_method': blend
    }


def process_all_scenes(method='SIFT', blend='linear'):
    """
    Process all 6 panorama scenes and generate summary statistics

    Args:
        method: Feature detector to use
        blend: Blending method

    Returns:
        List of result dictionaries for each scene
    """
    scenes = ['v_bird', 'v_boat', 'v_circus', 'v_graffiti', 'v_soldiers', 'v_weapons']
    results = []

    for scene in scenes:
        try:
            result = process_scene_complete(scene, method=method, blend=blend)
            results.append(result)
        except Exception as e:
            print(f" Error processing {scene}: {e}")
            results.append({'scene': scene, 'error': str(e)})

    # Print summary table
    print("PANORAMA PROCESSING SUMMARY")
    print(f"{'Scene':<15} {'KP1':<6} {'KP2':<6} {'Matches':<8} {'Inliers':<8} {'Ratio':<8} {'Error(px)':<10}")

    for r in results:
        if 'error' not in r:
            print(f"{r['scene']:<15} {r['keypoints'][0]:<6} {r['keypoints'][1]:<6} "
                  f"{r['matches']:<8} {r['inliers']:<8} {r['inlier_ratio']:<8.3f} {r['mean_error']:<10.2f}")
        else:
            print(f"{r['scene']:<15} FAILED: {r['error']}")

    return results

In [ ]:
# COMPARISON
def compare_detectors_all_scenes():
    """
    Compare SIFT vs ORB across all scenes


    This comparison helps justify detector choice in the report.
    """
    print("FEATURE DETECTOR COMPARISON: SIFT vs ORB")

    scenes = ['v_bird', 'v_boat', 'v_circus', 'v_graffiti', 'v_soldiers', 'v_weapons']
    methods = ['SIFT', 'ORB']
    all_results = []

    for scene in scenes:
        scene_path = os.path.join(PANORAMA_PATH, scene)

        # Find image files
        image_files = []
        for ext in ['.ppm', '.png', '.jpg', '.jpeg']:
            image_files.extend(sorted(glob.glob(os.path.join(scene_path, f'*{ext}'))))

        if len(image_files) < 2:
            continue

        # Load images
        img1 = cv2.imread(image_files[0])
        img2 = cv2.imread(image_files[1])
        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

        for method in methods:
            start_time = time.time()

            # Feature detection
            detector = FeatureDetector(method=method)
            kp1, desc1 = detector.detect_and_compute(gray1)
            kp2, desc2 = detector.detect_and_compute(gray2)

            # Feature matching
            matcher = FeatureMatcher(method=method)
            matches = matcher.match_features(desc1, desc2)

            # Homography + RANSAC
            pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
            pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

            estimator = HomographyEstimator()
            try:
                H, inliers = estimator.ransac_homography(pts1, pts2)
                inlier_ratio = np.sum(inliers) / len(matches)
            except:
                inlier_ratio = 0.0

            elapsed = time.time() - start_time

            all_results.append({
                'scene': scene,
                'method': method,
                'kp1': len(kp1),
                'kp2': len(kp2),
                'matches': len(matches),
                'inlier_ratio': inlier_ratio,
                'time': elapsed
            })

    # Print comparison table
    print(f"\n{'Scene':<15} {'Method':<8} {'KP1':<6} {'KP2':<6} {'Matches':<8} {'Inlier%':<9} {'Time(s)':<8}")
    print("-"*80)

    for r in all_results:
        print(f"{r['scene']:<15} {r['method']:<8} {r['kp1']:<6} {r['kp2']:<6} "
              f"{r['matches']:<8} {r['inlier_ratio']*100:<8.1f}% {r['time']:<8.3f}")

    # Compute average statistics per method
    print("AVERAGE STATISTICS")

    for method in methods:
        method_results = [r for r in all_results if r['method'] == method]
        avg_kp = np.mean([r['kp1'] for r in method_results])
        avg_matches = np.mean([r['matches'] for r in method_results])
        avg_inlier = np.mean([r['inlier_ratio'] for r in method_results]) * 100
        avg_time = np.mean([r['time'] for r in method_results])

        print(f"{method}:")
        print(f"  Average keypoints: {avg_kp:.0f}")
        print(f"  Average matches: {avg_matches:.0f}")
        print(f"  Average inlier ratio: {avg_inlier:.1f}%")
        print(f"  Average time: {avg_time:.3f}s")

    return all_results

In [ ]:
# AUGMENTED REALITY
class ARApplication:
    """
    Augmented Reality: Project video onto moving planar surface

    Challenge: Estimate per-frame homography to maintain geometric consistency

    Pipeline:
    1. For each frame of target video (book.mov):
       - Detect features in current frame and reference cover
       - Match features and estimate homography
       - Warp source video frame using homography
       - Composite warped frame onto book frame

    2. Handle temporal consistency:
       - Use previous homography as fallback if current estimation fails
       - Prevents sudden jumps or disappearance of overlay

    3. Aspect ratio handling:
       - Center-crop source video to match book cover aspect ratio
       - Ensures no distortion of overlaid content
    """

    def __init__(self, estimator, method='SIFT'):
        """
        Initialize AR application

        Args:
            estimator: HomographyEstimator instance
            method: Feature detector ('SIFT' or 'ORB')
        """
        self.estimator = estimator
        self.detector = FeatureDetector(method=method)
        self.matcher = FeatureMatcher(method=method, ratio_thresh=0.75)
        self.prev_H = None  # Store previous homography for temporal stability

    def crop_center(self, frame, target_ratio):
        """
        Center-crop frame to match target aspect ratio

        Ensures source video fits book cover without distortion

        Args:
            frame: Input video frame
            target_ratio: Desired width/height ratio

        Returns:
            Cropped frame
        """
        h, w = frame.shape[:2]
        current_ratio = w / h

        if current_ratio > target_ratio:
            # Frame is too wide, crop horizontally
            new_w = int(h * target_ratio)
            start = (w - new_w) // 2
            return frame[:, start:start+new_w]
        else:
            # Frame is too tall, crop vertically
            new_h = int(w / target_ratio)
            start = (h - new_h) // 2
            return frame[start:start+new_h, :]

    def process_ar_video(self, book_path, cover_path, source_path, output_path, save_samples=True):
        """
        Process complete AR video pipeline

        For each frame:
        1. Match cover -> current book frame
        2. Estimate H_cover2book
        3. Compute H_src2book = H_cover2book @ H_src2cover
        4. Warp source frame and composite

        Args:
            book_path: Path to target video (book.mov)
            cover_path: Path to reference cover image
            source_path: Path to source video to overlay
            output_path: Path to save output video
            save_samples: Whether to save representative frames
        """
        print("AUGMENTED REALITY VIDEO PROCESSING")

        # Load reference cover image
        cover = cv2.imread(cover_path)
        cover_gray = cv2.cvtColor(cover, cv2.COLOR_BGR2GRAY)
        h_cover, w_cover = cover.shape[:2]
        cover_ratio = w_cover / h_cover

        print(f" Cover dimensions: {w_cover}x{h_cover} (ratio: {cover_ratio:.2f})")

        # Open video captures
        cap_book = cv2.VideoCapture(book_path)
        cap_source = cv2.VideoCapture(source_path)

        # Get video properties
        fps = cap_book.get(cv2.CAP_PROP_FPS) or 25
        w_out = int(cap_book.get(cv2.CAP_PROP_FRAME_WIDTH))
        h_out = int(cap_book.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Use shorter video length (as per assignment instructions)
        max_frames = min(
            int(cap_book.get(cv2.CAP_PROP_FRAME_COUNT)),
            int(cap_source.get(cv2.CAP_PROP_FRAME_COUNT))
        )

        print(f" Output video: {w_out}x{h_out} @ {fps:.1f}fps, {max_frames} frames")

        # Create video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(output_path, fourcc, fps, (w_out, h_out))

        # Define cover corners for homography computation
        cover_corners = np.float32([
            [0, 0], [w_cover, 0], [w_cover, h_cover], [0, h_cover]
        ])

        frame_count = 0
        samples = []  # Store representative frames for visualization

        print("\nProcessing frames...")

        while frame_count < max_frames:
            # Read frames from both videos
            ret_b, book_frame = cap_book.read()
            ret_s, src_frame = cap_source.read()

            if not (ret_b and ret_s):
                break

            frame_count += 1

            # Progress indicator
            if frame_count % 30 == 0 or frame_count == 1:
                print(f"  Frame {frame_count}/{max_frames} ({100*frame_count/max_frames:.1f}%)")

            # Center-crop source frame to match cover aspect ratio
            src_cropped = self.crop_center(src_frame, cover_ratio)
            h_src, w_src = src_cropped.shape[:2]

            # Convert book frame to grayscale for feature detection
            book_gray = cv2.cvtColor(book_frame, cv2.COLOR_BGR2GRAY)

            # ===== Per-frame Homography Estimation =====

            # Detect features in cover and current book frame
            kp_cover, desc_cover = self.detector.detect_and_compute(cover_gray)
            kp_book, desc_book = self.detector.detect_and_compute(book_gray)

            # Check if detection succeeded
            if desc_cover is None or len(kp_cover) < 4:
                # Insufficient features in cover, use previous H or skip
                if self.prev_H is not None:
                    H_cover2book = self.prev_H
                else:
                    writer.write(book_frame)  # Write original frame
                    continue
            else:
                # Match features
                matches = self.matcher.match_features(desc_cover, desc_book)

                if len(matches) < 4:
                    # Insufficient matches, fallback to previous H
                    if self.prev_H is not None:
                        H_cover2book = self.prev_H
                    else:
                        writer.write(book_frame)
                        continue
                else:
                    # Extract matched points
                    pts_cover = np.float32([kp_cover[m.queryIdx].pt for m in matches])
                    pts_book = np.float32([kp_book[m.trainIdx].pt for m in matches])

                    # Estimate homography with RANSAC
                    try:
                        H_cover2book, _ = self.estimator.ransac_homography(pts_cover, pts_book)
                        self.prev_H = H_cover2book.copy()  # Store for next frame
                    except:
                        # RANSAC failed, use previous H
                        if self.prev_H is not None:
                            H_cover2book = self.prev_H
                        else:
                            writer.write(book_frame)
                            continue

            # ===== Warp Source Frame onto Book =====

            # Compute homography: source corners -> cover corners -> book frame
            src_corners = np.float32([
                [0, 0], [w_src, 0], [w_src, h_src], [0, h_src]
            ])

            # H_src2cover: maps source frame to cover dimensions
            H_src2cover = self.estimator.compute_homography_dlt(src_corners, cover_corners)

            # H_src2book: composition of transformations
            H_src2book = H_cover2book @ H_src2cover

            # Warp source frame
            warped_src = cv2.warpPerspective(src_cropped, H_src2book, (w_out, h_out))

            # Create mask for blending
            mask = np.ones((h_src, w_src), dtype=np.uint8) * 255
            warped_mask = cv2.warpPerspective(mask, H_src2book, (w_out, h_out))
            _, warped_mask = cv2.threshold(warped_mask, 1, 255, cv2.THRESH_BINARY)

            # ===== Composite onto Book Frame =====

            # Inverse mask for background
            inv_mask = cv2.bitwise_not(warped_mask)

            # Extract background (book without overlay region)
            bg = cv2.bitwise_and(book_frame, book_frame, mask=inv_mask)

            # Extract foreground (warped source)
            fg = cv2.bitwise_and(warped_src, warped_src, mask=warped_mask)

            # Combine
            result = cv2.add(bg, fg)

            # Write to output video
            writer.write(result)

            # Save representative frames for report
            if save_samples and frame_count in [10, max_frames//2, max_frames-10]:
                samples.append((frame_count, result.copy()))

        # Release resources
        cap_book.release()
        cap_source.release()
        writer.release()

        print(f"\n✓ AR video saved: {output_path}")
        print(f"✓ Total frames processed: {frame_count}")

        # Visualize representative frames
        if save_samples and samples:
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            for i, (fnum, frame) in enumerate(samples):
                axes[i].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                axes[i].set_title(f"Frame {fnum}")
                axes[i].axis('off')

                # Save individual frames
                frame_path = os.path.join(OUTPUT_PATH, 'ar_frames', f'ar_frame_{fnum}.jpg')
                cv2.imwrite(frame_path, frame)

            plt.tight_layout()
            composite_path = os.path.join(OUTPUT_PATH, 'ar_frames', 'ar_representative.jpg')
            plt.savefig(composite_path, dpi=150)
            plt.show()

            print(f"✓ Representative frames saved to: {OUTPUT_PATH}ar_frames/")


In [ ]:
# FULL EXECUTION
results = process_all_scenes(method='SIFT', blend='linear')
compare_detectors_all_scenes()

book_video = os.path.join(AR_PATH, "book.mov")
cover_image = os.path.join(AR_PATH, "cv_cover.jpg")
source_video = os.path.join(AR_PATH, "ar_source.mov")
output_video = os.path.join(OUTPUT_PATH, "ar_dynamic_result.mp4")

if all(os.path.exists(p) for p in [book_video, cover_image, source_video]):
    ar_app = ARApplication(HomographyEstimator(), method='SIFT')
    ar_app.process_ar_video(book_video, cover_image, source_video, output_video)
else:
    print("AR files missing!")

print("\nCOMPLETION SUMMARY")
print(f" Panoramas: {OUTPUT_PATH}panoramas/")
print(f" AR Video: {output_video}")
print(f" Report visuals in: {OUTPUT_PATH}")